In [ ]:
import pandas as pd
import os
import glob
import importlib
import config
import random

importlib.reload(config) #This is for reload the config if it is edited later

In [ ]:
# 1. Define a function to mask a column

def mask_column(dataframe, column_name, prefix="ID"):
    """
    Replaces real values with a unique ID.
    Example: 'Apple' becomes 'ID_1'
    """
    # Create a mapping of Unique Value -> New ID
    unique_values = {val: f"{prefix}_{i:03d}" for i, val in enumerate(dataframe[column_name].unique(), start=1)}
    
    # Apply the mapping to the column
    dataframe[column_name] = dataframe[column_name].map(unique_values)
    return unique_values

In [ ]:
# 2. Load Master Data and Daftar Saham (in Raw Data)

input_master = os.path.join(config.INGESTED_DATA_PATH, "Master_Raw_Data.csv")
saham_file = glob.glob(os.path.join(config.DAFTAR_SAHAM, "*.xlsx"))

try:
    if not saham_file:
        raise FileNotFoundError("No Excel files found in the Daftar_Saham folder.")

    input_saham = saham_file[0] 
    df_master = pd.read_csv(input_master)
    df_saham = pd.read_excel(input_saham)
    print(f"Successfully read: {os.path.basename(input_master)}")
    print(f"Successfully read: {os.path.basename(input_saham)}")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
print(f"\n{'='*30} GATEKEEPER {'='*30}\n")
print(df_master.head())
print(f"\n{'-' * 30} DATA SCHEMA {'-' * 30}")
print(df_master.info())

In [ ]:
print(f"\n{'='*30} GATEKEEPER {'='*30}\n")
print(df_saham.head())
print(f"\n{'-' * 30} DATA SCHEMA {'-' * 30}")
print(df_saham.info())

In [ ]:
# 3. Remove some columns that not needed

df_cleaned_master = df_master.copy()
df_cleaned_saham = df_saham.copy()

columns_to_remove_master = ["Nama Perusahaan", "Remarks"]
columns_to_remove_saham = ["Nama Perusahaan", "Tanggal Pencatatan"]

df_cleaned_master = df_cleaned_master.drop(columns=columns_to_remove_master)
df_cleaned_saham = df_cleaned_saham.drop(columns=columns_to_remove_saham)

print(f"\n{'-' * 30} Clean Master_Saham {'-' * 30}")
print(f"Columns after deletion: {df_cleaned_master.columns.tolist()}")
print(f"\n{'-' * 30} Clean Daftar Saham {'-' * 30}")
print(f"Columns after deletion: {df_cleaned_saham.columns.tolist()}")

In [ ]:
# 4. Apply masking

master_masked = df_cleaned_master.copy()
saham_masked = df_cleaned_saham.copy()

shared_map = mask_column(df_cleaned_saham.copy(), 'Kode', prefix="Saham")

if 'Kode Saham' in master_masked.columns:
    master_masked['Kode Saham'] = master_masked['Kode Saham'].map(shared_map)
    print("Masking complete for Master_Saham.")

if 'Kode' in saham_masked.columns:
    saham_masked['Kode'] = saham_masked['Kode'].map(shared_map)
    print("Masking complete for Daftar Saham.")

In [ ]:
print(f"\n{'-' * 30} Test Master_Saham {'-' * 30}")
print(master_masked["Kode Saham"])
print(f"\n{'-' * 30} Test Daftar Saham {'-' * 30}")
print(saham_masked["Kode"])

In [ ]:
# 5. Verify the masked

print(f"Empty rows in Master: {master_masked['Kode Saham'].isna().sum()}")
print(f"Empty rows in Daftar: {saham_masked['Kode'].isna().sum()}")

In [ ]:
# 6. Fill the empty rows with unknown

master_masked_final = master_masked.copy()
master_masked_final['Kode Saham'] = master_masked_final['Kode Saham'].fillna("Unknown_Saham")

print(f"Empty rows in Master: {master_masked_final['Kode Saham'].isna().sum()}")

In [ ]:
# 7. Verificate the stocks

target_id = 'Saham_932'
target_index = saham_masked[saham_masked['Kode'] == target_id].index[0]
original_name = df_cleaned_saham.iloc[target_index]['Kode']

print(f"--- Targeted Spot Check ---")
print(f"Masked ID:    {target_id}")
print(f"Row Index:    {target_index}")
print(f"Original Name: {original_name}")

if shared_map[original_name] == target_id:
    print(f"\n✅ VERIFIED: {original_name} is indeed {target_id}")

In [ ]:
# 8. Save the Masked Data

output_path_master = os.path.join(config.MASKED_DATA_RINGKASAN_PERDAGANGAN_SAHAM_PATH, "Master_Masked_Data.csv")
master_masked_final.to_csv(output_path_master, index=False)

output_path_daftar = os.path.join(config.MASKED_DATA_DAFTAR_SAHAM_PATH, "Daftar_Saham_Masked.csv")
saham_masked.to_csv(output_path_daftar, index=False)

print(f"Masked master file saved to: {output_path_master}")
print(f"Masked daftar saham file saved to: {output_path_daftar}")